In [21]:
!pip install edgartools earningscall



   ----- ---------------------------------- 1/8 [mypy-extensions]
   ---------- ----------------------------- 2/8 [marshmallow]
   --------------- ------------------------ 3/8 [cattrs]
   --------------- ------------------------ 3/8 [cattrs]
   --------------- ------------------------ 3/8 [cattrs]
   --------------- ------------------------ 3/8 [cattrs]
   ------------------------- -------------- 5/8 [requests-cache]
   ------------------------- -------------- 5/8 [requests-cache]
   ------------------------------ --------- 6/8 [dataclasses-json]
   ----------------------------------- ---- 7/8 [earningscall]
   ---------------------------------------- 8/8 [earningscall]



In [37]:
import numpy as np
import pandas as pd
import yfinance as yf
from edgar import *
from earningscall import get_company
from datetime import datetime,timedelta


In [2]:
set_identity("alaymajmudar.loyola@gmail.com")


In [94]:
import re

def clean_mda_narrative(raw_text):
    if not raw_text:
        return ""

    # 1. SPLIT INTO BLOCKS: Preserve the paragraph structure
    blocks = raw_text.split('\n')
    clean_blocks = []

    # 2. DEFINE DEBRIS PATTERNS (Common table headers/metadata)
    debris_keywords = [
        r'Three Months Ended', r'Six Months Ended', r'Percentage Change',
        r'In millions', r'In billions', r'Unaudited', r'Ended December',
        r'Fiscal Year', r'Consolidated Statements', r'Note [0-9]+'
    ]
    debris_regex = "|".join(debris_keywords)

    for block in blocks:
        # Normalize whitespace within the block
        block = re.sub(r'\s+', ' ', block).strip()
        
        # SKIP CRITERIA:
        # A. Block is too short (likely a stray header or date)
        if len(block) < 40:
            continue
            
        # B. Block matches common table debris
        if re.search(debris_regex, block, re.IGNORECASE):
            continue
            
        # C. Block is "Digit-Dense" (Even without numbers, tables have many symbols)
        symbol_count = len(re.findall(r'[\$\%\(\)\-\,]', block))
        if symbol_count > 5: # High symbol count usually indicates a table row
            continue

        # 3. NARRATIVE RECOVERY: Strip the numbers but keep the words
        # This preserves the "39% Azure growth" as "Azure growth"
        financial_pattern = r'\$?\b\(?\d{1,3}(?:,\d{3})*(?:\.\d+)?\b%?\)?'
        block = re.sub(financial_pattern, '', block)
        
        # 4. LEFTOVER CLEANUP: Remove empty brackets () and orphaned symbols
        block = re.sub(r'\(\s*\)', '', block) # Removes ()
        block = re.sub(r'\s[\-\,]\s', ' ', block) # Removes stray dashes/commas
        
        # 5. FINAL VALIDATION: Only keep blocks that look like actual sentences
        if len(block) > 40 and any(c.isalpha() for c in block):
            clean_blocks.append(block)

    # REJOIN: Use double newlines to keep paragraphs distinct for your RAG system
    return "\n\n".join(clean_blocks)

In [ ]:
filings = Company("MSFT").get_filings(form="10-Q").latest()

tenk = filings.obj().document
item_1a = tenk.get_section("Item 2")
if item_1a:
    # Note: Section objects also have text() method
    text = item_1a.text(include_tables=False,table_max_col_width=None)
    print(text)

In [95]:
modified_text = clean_mda_narrative(text)

In [ ]:
modified_text

In [98]:
filings.obj().reports

╭─────────────────────────────────────────── Reports ────────────────────────────────────────────╮
│                                                                                                │
│   #    Report                                                         Category       File      │
│  ────────────────────────────────────────────────────────────────────────────────────────────  │
│   1    Document and Entity Information                                Cover          R1.htm    │
│                                                                                                │
│   2    INCOME STATEMENTS                                              Statements     R2.htm    │
│                                                                                                │
│   3    COMPREHENSIVE INCOME STATEMENTS                                Statements     R3.htm    │
│                                                                                                │
│   4    B

In [60]:
with open("./10_Q.txt", "w") as file:
    file.write(filings.obj()['Item 2'])

# filings.obj()['Item 2']

In [22]:
for f in filings:
    print(f.parse())
    
    print(f.filing_date, f.form)

UNITED STATES

SECURITIES AND EXCHANGE COMMISSION

Washington, D.C. 20549

FORM10-Q

  QUARTERLY REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934   
  TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934  
 ────────────────────────────────────────────────────────────────────────────────────────────
  For the Quarterly Period Ended December 31, 2025                                          
  For the Transition Period From to                                                         

Commission File Number001-37845

MICROSOFT CORPORATION

  Washington                    91-1144442   
  (STATE OF INCORPORATION)      (I.R.S. ID)  

ONE MICROSOFT WAY, REDMOND, Washington98052-6399

(425)882-8080

www.microsoft.com/investor

  Securities registered pursuant to Section 12(b) of the Act:                                                      
  Title of each class                                              Trading Symbol      Name of exchange on which 
registered  
  Common stock, $0.00000625 par value per share                    MSFT                Nasdaq                      
  3.125% Notes due 2028                                            MSFT                Nasdaq                      
  2.625% Notes due 2033                                            MSFT                Nasdaq                      

Indicate by check mark whether the registrant (1) has filed all reports required to be filed by Section 13 or 15(d)
of the Securities Exchange Act of 1934 during the preceding 12 months (or for such shorter period that the 
registrant was required to file such reports), and (2) has been subject to such filing requirements for the past 90
days. Yes☒ No☐

Indicate by check mark whether the registrant has submitted electronically every Interactive Data File required to 
be submitted pursuant to Rule 405 of Regulation S-T (§232.405 of this chapter) during the preceding 12 months (or 
for such shorter period that the registrant was required to submit such files). Yes☒ No☐

Indicate by check mark whether the registrant is a large accelerated filer, an accelerated filer, a non-accelerated
filer, a smaller reporting company, or an emerging growth company. See the definitions of “large accelerated 
filer,” “accelerated filer,” “smaller reporting company,” and “emerging growth company” in Rule 12b-2 of the 
Exchange Act.

  Large Accelerated Filer ☒      Accelerated Filer ☐          
  Non-accelerated Filer ☐        Smaller Reporting Company ☐  
                                 Emerging Growth Company ☐    

If an emerging growth company, indicate by check mark if the registrant has elected not to use the extended 
transition period for complying with any new or revised financial accounting standards provided pursuant to Section
13(a) of the Exchange Act. ☐

Indicate by check mark whether the registrant is a shell company (as defined in Rule 12b-2 of the Exchange Act). 
Yes ☐ No☒

Indicate the number of shares outstanding of each of the issuer’s classes of common stock, as of the latest 
practicable date.

  Class                                               Outstanding as of January 22, 2026  
 ──────────────────────────────────────────────────────────────────────────────────────────
  Common Stock, $ 0.00000625 par value per share      7,425,629,076 shares                

MICROSOFT CORPORATION

FORM 10-Q

For the Quarter Ended December 31, 2025

INDEX

                                                                                                                   
                                                                                                                   
    Page  
  PART I.        FINANCIAL INFORMATION                                                                             
                 Item 1.                    Financial Statements                                                   
                                            a)          

None


In [6]:
sections = filings.sections()

In [31]:
sections

['10-K',
 'UNITED STATES',
 'SECURITIES AND EXCHANGE COMMISSION\nWashington, D.C. 20549',
 'FORM 10-K',
 '☒   ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934',
 'For the Fiscal Year Ended June 30, 2025                                                 \n    OR                                                                                      \n☐   TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934\n    For the Transition Period From to',
 'Commission File Number 001-37845',
 'MICROSOFT CORPORATION',
 'Washington                 91-1144442 \n(STATE OF INCORPORATION)   (I.R.S. ID)',
 'ONE MICROSOFT WAY, REDMOND, Washington 98052-6399\n(425) 882-8080\nwww.microsoft.com/investor',
 'Securities registered pursuant to Section 12(b) of the Act:                                                        \nTitle of each class                                           Trading Symbol   Name of exchange on which registered\nCom

TypeError: string indices must be integers, not 'tuple'

In [61]:
import re
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta
from edgar import Company, set_identity

# Required for SEC EDGAR access
set_identity("Alay Majmudar alaymajmudar.loyola@gmail.com")

def clean_tabular_noise(text):
    """Strips lines that are likely dense tables or numeric noise."""
    if not text: return ""
    lines = str(text).split('\n')
    # Filter out lines that have 4 or more numeric/financial symbols
    cleaned_lines = [line for line in lines if len(re.findall(r'\d|\$|%', line)) < 4]
    return " ".join(cleaned_lines).strip()

def ingest_sec_data(ticker="MSFT", years=5):
    """Fetches and cleans 10-K, 10-Q, and 8-K filings."""
    end_date = datetime(2026, 4, 20)
    start_date = end_date - timedelta(days=years*365)
    date_range = f"{start_date.strftime('%Y-%m-%d')}:{end_date.strftime('%Y-%m-%d')}"
    
    company = Company(ticker)
    # Fetching 8-K is crucial for real-time volatility triggers
    filings = company.get_filings(form=["10-K", "10-Q", "8-K"], filing_date=date_range)
    
    processed_docs = []
    for f in filings:
        try:
            doc = f.obj()
            entry = {"date": f.filing_date, "form": f.form, "accession": f.accession_number}
            
            # Using bracket notation to avoid AttributeError in TenQ
            if f.form == "10-K":
                entry["mda"] = doc['Item 7']
                entry["risk_factors"] = doc['Item 1A']
                entry["business_info"] = doc['Item 1'] # Future RAG context
            elif f.form == "10-Q":
                entry["mda"] = doc['Part I, Item 2'] # Fixed key for TenQ
                entry["risk_factors"] = doc['Part II, Item 1A']
            elif f.form == "8-K":
                entry["mda"] = f.text() # 8-Ks are often unstructured press releases
            
            # Apply tabular cleaning immediately
            entry["mda_clean"] = clean_tabular_noise(entry.get("mda", ""))
            entry["risk_clean"] = clean_tabular_noise(entry.get("risk_factors", ""))
            
            processed_docs.append(entry)
        except Exception as e:
            print(f"Error parsing {f.accession_number} ({f.form}): {e}")
            
    return pd.DataFrame(processed_docs)

def ingest_market_data(ticker="MSFT", years=5):
    """Fetches daily price data and the 10-Year Treasury yield."""
    tickers = [ticker, "^TNX", "^VIX"] # MSFT, 10Y Yield, and Volatility Index
    data = yf.download(tickers, start="2021-01-01", end="2026-04-20")
    
    # Flatten multi-index columns for easier SQL ingestion
    data.columns = ['_'.join(col).strip() for col in data.columns.values]
    return data.reset_index()

In [62]:
sec_data  = ingest_sec_data(years=1)

In [66]:
sec_data.head()

,date,form,accession,mda,risk_factors,mda_clean,risk_clean,business_info
0,2026-01-28,10-Q,0001193125-26-027207,ITEM 2. MANAGEMENT’S DISCUSSION AND ANALYSIS O...,ITEM 1A. RISK FACTORS\n\nOur operations and fi...,ITEM 2. MANAGEMENT’S DISCUSSION AND ANALYSIS O...,ITEM 1A. RISK FACTORS Our operations and fina...,NaN
1,2026-01-28,8-K,0001193125-26-027198,UNITED STATES\n\nSECURITIES AND EXCHANGE COMMI...,NaN,UNITED STATES SECURITIES AND EXCHANGE COMMISS...,,NaN
2,2025-12-08,8-K,0001193125-25-311196,UNITED STATES\n\nSECURITIES AND EXCHANGE COMMI...,NaN,UNITED STATES SECURITIES AND EXCHANGE COMMISS...,,NaN
3,2025-10-29,10-Q,0001193125-25-256321,ITEM 2. MANAGEMENT’S DISCUSSION AND ANALYSIS O...,ITEM 1A. RISK FACTORS\n\nOur operations and fi...,ITEM 2. MANAGEMENT’S DISCUSSION AND ANALYSIS O...,ITEM 1A. RISK FACTORS Our operations and fina...,NaN
4,2025-10-29,8-K,0001193125-25-256310,UNITED STATES\n\nSECURITIES AND EXCHANGE COMMI...,NaN,UNITED STATES SECURITIES AND EXCHANGE COMMISS...,,NaN


In [69]:
sec_data.loc[0,'mda']

'ITEM 2. MANAGEMENT’S DISCUSSION AND ANALYSIS OF FINANCIAL CONDITION AND RESULTS OF OPERATIONS\n\nNote About Forward-Looking Statements\n\nThis report includes estimates, projections, statements relating to our business plans, objectives, and expected operating results that are “forward-looking statements” within the meaning of the Private Securities Litigation Reform Act of 1995, Section 27A of the Securities Act of 1933, and Section 21E of the Securities Exchange Act of 1934. Forward-looking statements may appear throughout this report, including the following sections: “Management’s Discussion and Analysis of Financial Condition and Results of Operations” and “Risk Factors” (Part II, Item 1A of this Form 10-Q). These forward-looking statements generally are identified by the words “believe,” “project,” “expect,” “anticipate,” “estimate,” “intend,” “strategy,” “future,” “opportunity,” “plan,” “may,” “should,” “will,” “would,” “will be,” “will continue,” “will likely result,” and simi

In [44]:
df_filings['risk_factors'][:1]

0    NaN
Name: risk_factors, dtype: str

In [132]:
'Q&A'.lower()

'q&a'

In [147]:


company = get_company("MSFT")  # Lookup Apple, Inc by its ticker symbol, "AAPL"

transcript = company.get_transcript(year=2025, quarter=2,level=2)
print(transcript.text)
# print(f"{company} Q3 2021 Transcript Text: \"{transcript.text[:1000]}...\"")



Greetings and welcome to the Microsoft fiscal year 2025 second quarter earnings conference call. At this time, all participants are in a listen-only mode. A question and answer session will follow the formal presentation. If anyone should require operator assistance, please press star zero on your telephone keypad. As a reminder, this conference is being recorded. It is now my pleasure to introduce your host, Brett Iverson, Vice President of Investor Relations. Please go ahead. Good afternoon, and thank you for joining us today. On the call with me are Satya Nadella, Chairman and Chief Executive Officer, Amy Hood, Chief Financial Officer, Alice Jolla, Chief Accounting Officer, and Keith Dolliver, Corporate Secretary and Deputy General Counsel. On the Microsoft Investor Relations website, you can find our earnings press release and financial summary slide deck, which is intended to supplement our prepared remarks during today's call and provides the reconciliation of differences between

In [165]:
from earningscall import get_company
def process_structured_transcript(company_ticker, year, quarter):
    company = get_company(company_ticker)
    
    # LEVEL 2 gives us the names and the text in order
    transcript = company.get_transcript(year=year, quarter=quarter, level=2)
    
    prepared_remarks = []
    qa_pairs = []
    is_qa_started = False
    qa_markers = ["q and a", "q&a"]
    
    for turn in transcript.speakers:
        speaker_name = turn.speaker_info.name.strip()
        text = turn.text.strip()

        # 1. DETECT THE PIVOT: The Operator turn that opens the floor
        if any(marker in text.lower() for marker in qa_markers):
            is_qa_started = True
            continue # Skip the operator's intro block
        
        # 2. CATEGORIZE THE TURN
        if not is_qa_started:
            prepared_remarks.append({"speaker": speaker_name, "text": text})
        else:
            qa_pairs.append({"speaker": speaker_name, "text": text})

    return prepared_remarks, qa_pairs

In [168]:
p,q = process_structured_transcript('MSFT','2025',2)

In [ ]:
p

In [172]:
import earningscall
from datetime import datetime, timedelta


def process_msft_history(ticker="MSFT"):
    company = earningscall.get_company(ticker)
    
    # 1. Define the 5-year boundary
    five_years_ago = datetime.now() - timedelta(days=5*365)
    
    # Storage for all processed quarters
    historical_data = []

    # 2. Iterate through events
    for event in company.events():
        # Skip future events
        if datetime.now() < event.conference_date:
            continue
            
        # Filter for the last 5 years
        if event.conference_date < five_years_ago:
            continue

        print(f"Ingesting: Q{event.quarter} {event.year} (Date: {event.conference_date.strftime('%Y-%m-%d')})")
        
        try:
            # Fetch Level 2 to get speaker names and text
            transcript = company.get_transcript(event=event, level=2)
            
            if not transcript or not transcript.speakers:
                continue

            # 3. Apply the split logic to create our lists of dicts
            prepped, qa = split_transcript_by_turns(transcript.speakers)
            
            # Store results with metadata for the training table
            historical_data.append({
                "date": event.conference_date.strftime('%Y-%m-%d'),
                "year": event.year,
                "quarter": event.quarter,
                "prepped_list": prepped,
                "qa_list": qa
            })

        except Exception as e:
            print(f"Error processing Q{event.quarter} {event.year}: {e}")

    return historical_data

def split_transcript_by_turns(turns):
    """
    Processes Level 2 turns into two lists of dictionaries based on the Q&A pivot.
    """
    prepped_list = []
    qa_list = []
    is_qa_started = False
    
    # MSFT specific pivot markers
    qa_markers = ["q and a", "q&a"]

    for turn in turns:
        speaker = turn.speaker if turn.speaker else "Unknown"
        text = turn.text if turn.text else ""
        text_lower = text.lower()

        # Detect transition
        if not is_qa_started:
            if any(marker in text_lower for marker in qa_markers):
                is_qa_started = True
                # We skip the specific turn that announces the Q&A to keep data clean
                continue 

        # Add to respective list as a dictionary
        entry = {"speaker": speaker, "text": text}
        
        if is_qa_started:
            qa_list.append(entry)
        else:
            prepped_list.append(entry)

    return prepped_list, qa_list

# Execute
msft_dataset = process_msft_history()

TypeError: can't compare offset-naive and offset-aware datetimes

In [2]:
import sys
print(sys.version)

3.14.4 | packaged by Anaconda, Inc. | (main, Apr 14 2026, 17:00:17) [MSC v.1942 64 bit (AMD64)]


In [3]:
dat = yf.Ticker("MSFT")


In [9]:
dat.info()

YFRateLimitError: Too Many Requests. Rate limited. Try after a while.